# 17 - Strict Split LR Fusion

Notebook propuesto para corregir la seleccion de inferencia y evitar mezcla de idiomas.

Mejoras incluidas:
- Split/inferencia estricto por idioma (sin fallback global).
- Fusion robusta texto + video + sensorial con Logistic Regression.
- Reporte CV macro-F1 y export JSON por ES, EN y ES_EN.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

SEED = 42
GROUP_ID = 'BeingChillingWeWillWin'
MODEL_PREFIX = '17StrictLRFusion'
TEST_CASE = 'EXIST2026'

np.random.seed(SEED)


In [2]:
def resolve_notebooks_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd] + list(cwd.parents)
    for c in candidates:
        if (c / 'artifacts').exists() and (c / 'entregables').exists():
            return c
        if (c / 'notebooks_shiyi' / 'artifacts').exists():
            return c / 'notebooks_shiyi'
    raise FileNotFoundError('Could not resolve notebooks_shiyi root')

ROOT = resolve_notebooks_root()
ARTIFACTS = ROOT / 'artifacts'
ENTREGABLES = ROOT / 'entregables'
REPORTS = ROOT / 'reports_hpo'
ENTREGABLES.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

train_candidates = [
    ROOT.parent / 'materials' / 'dataset_task3_exist2026' / 'training.json',
    Path('/home/alumno.upv.es/scheng1/EXIST 2026 Videos Dataset/training/training.json'),
]
TRAIN_JSON = next((p for p in train_candidates if p.exists()), None)
if TRAIN_JSON is None:
    raise FileNotFoundError('training.json not found in local or cluster paths')

print('ROOT:', ROOT)
print('TRAIN_JSON:', TRAIN_JSON)


ROOT: /home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/notebooks_shiyi
TRAIN_JSON: /home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/materials/dataset_task3_exist2026/training.json


In [3]:
def majority_task3_1(labels):
    if isinstance(labels, list):
        vals = [x for x in labels if x in {'YES', 'NO'}]
        if not vals:
            return None
        yes = sum(v == 'YES' for v in vals)
        no = sum(v == 'NO' for v in vals)
        return 'YES' if yes > no else 'NO'
    if labels in {'YES', 'NO'}:
        return labels
    return None

raw = json.loads(TRAIN_JSON.read_text(encoding='utf-8'))
rows = []
for sid, payload in raw.items():
    rec = {'sample_id': str(sid)}
    rec.update(payload)
    rows.append(rec)
df_raw = pd.DataFrame(rows)
df_raw['text'] = df_raw['text'].fillna('').astype(str)

meta_path = ARTIFACTS / 'sample_metadata.csv'
if meta_path.exists():
    meta = pd.read_csv(meta_path)
    meta['sample_id'] = meta['sample_id'].astype(str)
else:
    meta = pd.DataFrame({'sample_id': df_raw['sample_id'].astype(str)})

if 'y' not in meta.columns or meta['y'].isna().all():
    y_map = df_raw['labels_task3_1'].apply(majority_task3_1).map({'NO': 0, 'YES': 1})
    meta = meta.merge(df_raw[['sample_id']].assign(y=y_map.values), on='sample_id', how='left')

video = pd.read_csv(ARTIFACTS / 'video_features.csv')
sensor = pd.read_csv(ARTIFACTS / 'sensorial_features.csv')
video['sample_id'] = video['sample_id'].astype(str)
sensor['sample_id'] = sensor['sample_id'].astype(str)

df = meta[['sample_id', 'lang', 'split', 'y']].copy()
for col in ['lang', 'split']:
    if col not in df.columns and col in df_raw.columns:
        df[col] = df_raw.set_index('sample_id')[col].reindex(df['sample_id']).values

df = df.merge(df_raw[['sample_id', 'text']], on='sample_id', how='left')
df = df.merge(video, on='sample_id', how='left')
df = df.merge(sensor, on='sample_id', how='left')

df['text'] = df['text'].fillna('').astype(str)
df['lang'] = df['lang'].fillna('').astype(str).str.lower()
df['y'] = pd.to_numeric(df['y'], errors='coerce').fillna(-1).astype(int)

num_cols = [
    c for c in df.columns
    if c.startswith('video__')
    or c.startswith('et__')
    or c.startswith('eeg__')
    or c.startswith('hr__')
    or c.startswith('eda__')
    or c.startswith('ecg__')
]
if not num_cols:
    num_cols = [c for c in df.columns if c not in {'sample_id', 'lang', 'split', 'text', 'y'}]

df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce').fillna(0.0)

print('Merged shape:', df.shape)
print('Languages:', df['lang'].value_counts(dropna=False).to_dict())
print('Numeric cols:', len(num_cols))


Merged shape: (2006, 121)
Languages: {'es': 1212, 'en': 794}
Numeric cols: 116


In [4]:
def export_json(sample_ids, preds, out_path: Path):
    label_map = {0: 'NO', 1: 'YES'}
    rows = [
        {'test_case': TEST_CASE, 'id': str(sid), 'value': label_map[int(p)]}
        for sid, p in zip(sample_ids, preds)
    ]
    with out_path.open('w', encoding='utf-8') as f:
        json.dump(rows, f, ensure_ascii=False)

def run_scope(scope_name: str, langs):
    if langs is None:
        scope_df = df.copy()
    else:
        scope_df = df[df['lang'].isin(langs)].copy()

    train_df = scope_df[scope_df['y'].isin([0, 1])].copy()
    if train_df.empty:
        raise RuntimeError(f'No training rows for scope={scope_name}')

    split_upper = scope_df['split'].fillna('').astype(str).str.upper()
    test_mask = split_upper.str.contains('TEST')
    infer_df = scope_df.loc[test_mask].copy() if test_mask.any() else scope_df.copy()

    if infer_df.empty:
        raise RuntimeError(f'No inference rows for scope={scope_name}')

    if langs is not None:
        bad_langs = sorted(set(infer_df['lang'].unique()) - set(langs))
        if bad_langs:
            raise RuntimeError(f'Language leak in scope={scope_name}: {bad_langs}')

    tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    scaler = StandardScaler()

    X_text_train = tfidf.fit_transform(train_df['text'].astype(str))
    X_text_infer = tfidf.transform(infer_df['text'].astype(str))

    X_num_train = scaler.fit_transform(train_df[num_cols].to_numpy(dtype=np.float32))
    X_num_infer = scaler.transform(infer_df[num_cols].to_numpy(dtype=np.float32))

    X_train = hstack([X_text_train, csr_matrix(X_num_train)], format='csr')
    X_infer = hstack([X_text_infer, csr_matrix(X_num_infer)], format='csr')
    y_train = train_df['y'].to_numpy(dtype=np.int64)

    binc = np.bincount(y_train)
    if len(binc) < 2 or np.min(binc) < 2:
        raise RuntimeError(f'Not enough class support for CV in scope={scope_name}')
    n_splits = int(max(2, min(5, np.min(binc))))

    model = LogisticRegression(
        max_iter=3000,
        class_weight='balanced',
        solver='liblinear',
        random_state=SEED,
    )

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)

    model.fit(X_train, y_train)
    infer_prob = model.predict_proba(X_infer)[:, 1]
    infer_pred = (infer_prob >= 0.5).astype(int)

    out_json = ENTREGABLES / f'{GROUP_ID}_{MODEL_PREFIX}_{scope_name}.json'
    export_json(infer_df['sample_id'].astype(str).tolist(), infer_pred, out_json)

    train_pred = model.predict(X_train)
    train_f1 = float(f1_score(y_train, train_pred, average='macro'))

    return {
        'scope': scope_name,
        'rows_train': int(len(train_df)),
        'rows_infer': int(len(infer_df)),
        'cv_f1_macro_mean': float(np.mean(cv_scores)),
        'cv_f1_macro_std': float(np.std(cv_scores)),
        'train_f1_macro': train_f1,
        'json_path': str(out_json),
    }

results = []
for scope_name, langs in [('ES', ['es']), ('EN', ['en']), ('ES_EN', None)]:
    r = run_scope(scope_name, langs)
    results.append(r)
    print(scope_name, 'cv_f1=', r['cv_f1_macro_mean'], 'rows_infer=', r['rows_infer'])

report_df = pd.DataFrame(results).sort_values('cv_f1_macro_mean', ascending=False)
report_path = REPORTS / '17_strict_lr_fusion_cv.csv'
report_df.to_csv(report_path, index=False)
print('Report saved:', report_path)
report_df


ES cv_f1= 0.6338258667295291 rows_infer= 1212


EN cv_f1= 0.6192126673200843 rows_infer= 794


ES_EN cv_f1= 0.621569530255895 rows_infer= 2006
Report saved: /home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/notebooks_shiyi/reports_hpo/17_strict_lr_fusion_cv.csv


,scope,rows_train,rows_infer,cv_f1_macro_mean,cv_f1_macro_std,train_f1_macro,json_path
0,ES,1212,1212,0.633826,0.013637,0.871245,/home/alumno.upv.es/scheng1/Master-IA-ALC/lab3...
2,ES_EN,2006,2006,0.621570,0.014778,0.870609,/home/alumno.upv.es/scheng1/Master-IA-ALC/lab3...
1,EN,794,794,0.619213,0.027564,0.884664,/home/alumno.upv.es/scheng1/Master-IA-ALC/lab3...
